In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mplhep as hep
import gc 

In [8]:
# Definir las ramas
ramas_el = ['Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dz', 'Electron_dxyErr', 'Electron_dzErr']
ramas_mu = ['Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_charge', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr']

# Ramas combinadas para los canales mixtos
ramas_Higgselmu = ramas_el + ramas_mu
ramas_2mu2el = ramas_el + ramas_mu

# Señal MC: H → ZZ* → 4l
file_higgs_2e2mu = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/SMHiggsToZZTo4L.root")
tree_higgs_2e2mu = file_higgs_2e2mu['Events']

# Fondo MC: ZZ → 4e
file_zz_4e = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/ZZTo4e.root")
tree_zz_4e = file_zz_4e['Events']

# Fondo MC: ZZ → 4mu
file_zz_2mu = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/ZZTo4mu.root")
tree_zz_2mu = file_zz_2mu['Events']

# Fondo MC: ZZ → 2e2mu
file_zz_2e2mu = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/ZZTo2e2mu.root")
tree_zz_2e2mu = file_zz_2e2mu['Events']


In [10]:
# Simulaciones (MC) 
df_higgs = tree_higgs_2e2mu.arrays(ramas_Higgselmu, library="ak")    
df_zz_4e = tree_zz_4e.arrays(ramas_el, library="ak") 
df_zz_2mu = tree_zz_2mu.arrays(ramas_mu, library="ak")      
df_zz_2e2mu = tree_zz_2e2mu.arrays(ramas_2mu2el, library="ak")  

print("Entradas totales en Higgs (MC):", tree_higgs_2e2mu.num_entries)
print("Entradas totales en ZZ -> 4e:", tree_zz_4e.num_entries)
print("Entradas totales en ZZ -> 4mu:", tree_zz_2mu.num_entries)
print("Entradas totales en ZZ -> 2e2mu:", tree_zz_2e2mu.num_entries)

Entradas totales en Higgs (MC): 299973
Entradas totales en ZZ -> 4e: 1499093
Entradas totales en ZZ -> 4mu: 1499064
Entradas totales en ZZ -> 2e2mu: 1497445


In [12]:
# Cargamos los datos reales limpios desde los archivos parquet
df_data_2el = ak.from_parquet("df_data_2el.parquet")
df_data_2mu = ak.from_parquet("df_data_2mu.parquet")

print("Entradas en df_data_2el:", len(df_data_2el))
print("Entradas en df_data_2mu:", len(df_data_2mu))


Entradas en df_data_2el: 14723467
Entradas en df_data_2mu: 28229800


In [15]:
#  Aplicamos máscara de multiplicidad para los electrones 
mask_2el = ak.num(df_data_2el["Electron_pt"]) >= 2
filtered_2el = df_data_2el[mask_2el]

# Creamos el DataFrame de Pandas extrayendo la pareja (índices 0 y 1)
df_par_2el = pd.DataFrame({
    "pt1": ak.to_numpy(filtered_2el["Electron_pt"][:, 0]),
    "eta1": ak.to_numpy(filtered_2el["Electron_eta"][:, 0]),
    "phi1": ak.to_numpy(filtered_2el["Electron_phi"][:, 0]),
    "charge1": ak.to_numpy(filtered_2el["Electron_charge"][:, 0]),
    "iso1": ak.to_numpy(filtered_2el["Electron_pfRelIso03_all"][:, 0]),
    
    "pt2": ak.to_numpy(filtered_2el["Electron_pt"][:, 1]),
    "eta2": ak.to_numpy(filtered_2el["Electron_eta"][:, 1]),
    "phi2": ak.to_numpy(filtered_2el["Electron_phi"][:, 1]),
    "charge2": ak.to_numpy(filtered_2el["Electron_charge"][:, 1]),
    "iso2": ak.to_numpy(filtered_2el["Electron_pfRelIso03_all"][:, 1]),
})
print ( "Eventos con 2 electrones extraídos:", len(df_par_2el))

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\awkward\_nplikes\array_module.py:375: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  result.shape = shape


Eventos con 2 electrones extraídos: 14723467


In [18]:
#  Aplicamos máscara de multiplicidad para los muones 
mask_mu = ak.num(df_data_2mu["Muon_pt"]) >= 2
filtered_mu = df_data_2mu[mask_mu]

df_par_2mu = pd.DataFrame({
    "pt1": ak.to_numpy(filtered_mu["Muon_pt"][:, 0]),
    "eta1": ak.to_numpy(filtered_mu["Muon_eta"][:, 0]),
    "phi1": ak.to_numpy(filtered_mu["Muon_phi"][:, 0]),
    "charge1": ak.to_numpy(filtered_mu["Muon_charge"][:, 0]),
    "dxy1": ak.to_numpy(filtered_mu["Muon_dxy"][:, 0]),
    "dz1": ak.to_numpy(filtered_mu["Muon_dz"][:, 0]),
    
    "pt2": ak.to_numpy(filtered_mu["Muon_pt"][:, 1]),
    "eta2": ak.to_numpy(filtered_mu["Muon_eta"][:, 1]),
    "phi2": ak.to_numpy(filtered_mu["Muon_phi"][:, 1]),
    "charge2": ak.to_numpy(filtered_mu["Muon_charge"][:, 1]),
    "dxy2": ak.to_numpy(filtered_mu["Muon_dxy"][:, 1]),
    "dz2": ak.to_numpy(filtered_mu["Muon_dz"][:, 1]),
})
print ( "Eventos con 2 muones extraídos:", len(df_par_2mu))

Eventos con 2 muones extraídos: 28229800


In [22]:
mask_higgs = (ak.num(df_higgs["Electron_pt"]) >= 2) & (ak.num(df_higgs["Muon_pt"]) >= 2)
filtered_higgs = df_higgs[mask_higgs]

df_par_higgs = pd.DataFrame({
    # Par de electrones
    "el_pt1": ak.to_numpy(filtered_higgs["Electron_pt"][:, 0]),
    "el_eta1": ak.to_numpy(filtered_higgs["Electron_eta"][:, 0]),
    "el_phi1": ak.to_numpy(filtered_higgs["Electron_phi"][:, 0]),
    "el_charge1": ak.to_numpy(filtered_higgs["Electron_charge"][:, 0]),
    
    "el_pt2": ak.to_numpy(filtered_higgs["Electron_pt"][:, 1]),
    "el_eta2": ak.to_numpy(filtered_higgs["Electron_eta"][:, 1]),
    "el_phi2": ak.to_numpy(filtered_higgs["Electron_phi"][:, 1]),
    "el_charge2": ak.to_numpy(filtered_higgs["Electron_charge"][:, 1]),
    
    # Par de muones
    "mu_pt1": ak.to_numpy(filtered_higgs["Muon_pt"][:, 0]),
    "mu_eta1": ak.to_numpy(filtered_higgs["Muon_eta"][:, 0]),
    "mu_phi1": ak.to_numpy(filtered_higgs["Muon_phi"][:, 0]),
    "mu_charge1": ak.to_numpy(filtered_higgs["Muon_charge"][:, 0]),
    
    "mu_pt2": ak.to_numpy(filtered_higgs["Muon_pt"][:, 1]),
    "mu_eta2": ak.to_numpy(filtered_higgs["Muon_eta"][:, 1]),
    "mu_phi2": ak.to_numpy(filtered_higgs["Muon_phi"][:, 1]),
    "mu_charge2": ak.to_numpy(filtered_higgs["Muon_charge"][:, 1]),
})

In [ ]:
#  Aplicamos máscara de multiplicidad para los monte carlo 
mask_zz4e = ak.num(df_zz_4e["Electron_pt"]) >= 2

filtered_zz4e = df_zz_4e[mask_zz4e]

df_par_zz4e = pd.DataFrame({
    "pt1": ak.to_numpy(filtered_zz4e["Electron_pt"][:, 0]),
    "eta1": ak.to_numpy(filtered_zz4e["Electron_eta"][:, 0]),
    "phi1": ak.to_numpy(filtered_zz4e["Electron_phi"][:, 0]),
    "charge1": ak.to_numpy(filtered_zz4e["Electron_charge"][:, 0]),
    "iso1": ak.to_numpy(filtered_zz4e["Electron_pfRelIso03_all"][:, 0]),
    
    "pt2": ak.to_numpy(filtered_zz4e["Electron_pt"][:, 1]),
    "eta2": ak.to_numpy(filtered_zz4e["Electron_eta"][:, 1]),
    "phi2": ak.to_numpy(filtered_zz4e["Electron_phi"][:, 1]),
    "charge2": ak.to_numpy(filtered_zz4e["Electron_charge"][:, 1]),
    "iso2": ak.to_numpy(filtered_zz4e["Electron_pfRelIso03_all"][:, 1]),
})



In [20]:
mask_zz4mu = ak.num(df_zz_2mu["Muon_pt"]) >= 2
filtered_zz4mu = df_zz_2mu[mask_zz4mu]

df_par_zz4mu = pd.DataFrame({
    "pt1": ak.to_numpy(filtered_zz4mu["Muon_pt"][:, 0]),
    "eta1": ak.to_numpy(filtered_zz4mu["Muon_eta"][:, 0]),
    "phi1": ak.to_numpy(filtered_zz4mu["Muon_phi"][:, 0]),
    "charge1": ak.to_numpy(filtered_zz4mu["Muon_charge"][:, 0]),
    "dxy1": ak.to_numpy(filtered_zz4mu["Muon_dxy"][:, 0]),
    "dz1": ak.to_numpy(filtered_zz4mu["Muon_dz"][:, 0]),
    
    "pt2": ak.to_numpy(filtered_zz4mu["Muon_pt"][:, 1]),
    "eta2": ak.to_numpy(filtered_zz4mu["Muon_eta"][:, 1]),
    "phi2": ak.to_numpy(filtered_zz4mu["Muon_phi"][:, 1]),
    "charge2": ak.to_numpy(filtered_zz4mu["Muon_charge"][:, 1]),
    "dxy2": ak.to_numpy(filtered_zz4mu["Muon_dxy"][:, 1]),
    "dz2": ak.to_numpy(filtered_zz4mu["Muon_dz"][:, 1]),
})

In [21]:
mask_zz2e2mu = (ak.num(df_zz_2e2mu["Electron_pt"]) >= 2) & (ak.num(df_zz_2e2mu["Muon_pt"]) >= 2)
filtered_zz2e2mu = df_zz_2e2mu[mask_zz2e2mu]

df_par_zz2e2mu = pd.DataFrame({
    # Par de electrones
    "el_pt1": ak.to_numpy(filtered_zz2e2mu["Electron_pt"][:, 0]),
    "el_eta1": ak.to_numpy(filtered_zz2e2mu["Electron_eta"][:, 0]),
    "el_phi1": ak.to_numpy(filtered_zz2e2mu["Electron_phi"][:, 0]),
    "el_charge1": ak.to_numpy(filtered_zz2e2mu["Electron_charge"][:, 0]),
    
    "el_pt2": ak.to_numpy(filtered_zz2e2mu["Electron_pt"][:, 1]),
    "el_eta2": ak.to_numpy(filtered_zz2e2mu["Electron_eta"][:, 1]),
    "el_phi2": ak.to_numpy(filtered_zz2e2mu["Electron_phi"][:, 1]),
    "el_charge2": ak.to_numpy(filtered_zz2e2mu["Electron_charge"][:, 1]),
    
    # Par de muones
    "mu_pt1": ak.to_numpy(filtered_zz2e2mu["Muon_pt"][:, 0]),
    "mu_eta1": ak.to_numpy(filtered_zz2e2mu["Muon_eta"][:, 0]),
    "mu_phi1": ak.to_numpy(filtered_zz2e2mu["Muon_phi"][:, 0]),
    "mu_charge1": ak.to_numpy(filtered_zz2e2mu["Muon_charge"][:, 0]),
    
    "mu_pt2": ak.to_numpy(filtered_zz2e2mu["Muon_pt"][:, 1]),
    "mu_eta2": ak.to_numpy(filtered_zz2e2mu["Muon_eta"][:, 1]),
    "mu_phi2": ak.to_numpy(filtered_zz2e2mu["Muon_phi"][:, 1]),
    "mu_charge2": ak.to_numpy(filtered_zz2e2mu["Muon_charge"][:, 1]),
})

In [23]:

# Guardamos cada DataFrame de parejas en formato Apache Parquet
df_par_2el.to_parquet("df_par_el_limpio.parquet")
df_par_2mu.to_parquet("df_par_mu_limpio.parquet")
df_par_zz4e.to_parquet("df_par_zz4e_limpio.parquet")
df_par_zz4mu.to_parquet("df_par_zz4mu_limpio.parquet")
df_par_higgs.to_parquet("df_par_higgs_limpio.parquet")
df_par_zz2e2mu.to_parquet("df_par_zz2e2mu_limpio.parquet")

print("¡Todos los DataFrames de parejas han sido guardados en archivos Parquet con éxito!")

¡Todos los DataFrames de parejas han sido guardados en archivos Parquet con éxito!
